# **EMG_Signal_Measurement_Implementation_Unit(III)**

> In this experiement, we will use FFT to observe EMG signal's frequency domain.

> And attempt on using MF and MPF to analysis our own EMG signal. Using hard data to see if your muscle is fatigued!

# 0. First import some formulas and functions we will be using!

In [ ]:
#@title
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
def butter_bandpass(low_cut, high_cut, fs, order=3):
    """
    Design band pass filter.

    Args:
        - low_cut  (float) : the low cutoff frequency of the filter.
        - high_cut (float) : the high cutoff frequency of the filter.
        - fs       (float) : the sampling rate.
        - order      (int) : order of the filter, by default defined to 5.
    """
    # calculate the Nyquist frequency
    nyq = 0.5 * fs

    # design filter
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype='band')

    # returns the filter coefficients: numerator and denominator
    return b, a

from scipy.signal import butter, lfilter
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

# 1. Enter your EMG signal of continuous muscle tension into the codebase!

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
df = pd.read_csv('Sample_EMG_Signal2.txt')  #Remember to change the file name in ('xxxx')!
df.info()
data = np.array(df)
data2 = data[0 : , 0]

# 2. Plot out the unprocessed EMG signal to see what it looks like!

In [ ]:
#Config the figure setting!
start_time = 2 #Unit: seconds
end_time = 8 #Unit: seconds

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2[start_time*1000:end_time*1000], #Sample a section of EMG signal
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="A section of EMG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 3. According to the thesis, use band-pass filter to retrieve the signal in range (20Hz~400Hz)!

In [ ]:
data3 = butter_bandpass_filter(data2, 20, 400, 1000)

start_time = 2 #Unit: seconds
end_time = 8 #Unit: seconds

#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data3[start_time*1000:end_time*1000], #Sample a section of EMG signal
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="Filtered EMG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 4. Retrieve MF and MPF from the filtered EMG signal using FFT!

> To observe the changes of MF and MPF when muscle are being tensioned, we take 2 seconds(2000 miliseconds) of data as a unit and use FFT to retrieve the frequency domain.

> After getting the frequency domain, try to calculate MF (median frequency) and MPF (mean power frequency) from the data segment!

In [ ]:
N = 2000 #Grab 2000 data points to do FFT (for data of 1000Hz, 1 second of data gives 1000 data points)
T = 0.001 #The interval between each data point is 0.001 second (for data of 1000Hz, data point interval is 0.001 second)
x = np.linspace(0.0, N*T, N, endpoint=False)
yf = fft(data3[1:N])
xf = fftfreq(N, T)[1:N//2]
yf_abs = np.abs(yf[1:N//2])
fig = go.Figure()
fig.add_trace( go.Scatter(x=xf, y=2.0/N * yf_abs))
fig.update_layout(
    title="Frequency domain energy data after taking FFT of the EMG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
#According to the formula, most important thing in doing MF is to know the sum of the frequency domain energy!
#Then we just sum it again one by one to easily get the median frquency position!

halfSum = 0 #Prepare to sum up to half of the total energy
MF = 0 #Calculate the position of MF in the frequency domain
fullSum = np.sum(yf_abs) #Sum of all the energy

for i in yf_abs:
  if(halfSum < fullSum/2): #When we add up to halfSum we arrive at MF's position (half of total energy)
    halfSum = halfSum + i
    MF += 1
  else:
    break
whole_MF = MF
print('The median frequency of the entire dataset is', whole_MF,'Hz',)

In [ ]:
#According to the formula, doing MPF is to get weighted average on all the frequency using their power level as weight!

fullSum = np.sum(yf_abs) #Sum of all the energy
frequency_pointer = 0 #Pointer to get the frequency domain data
MPF = 0 #Prepare to sum up to total energy

for i in yf_abs:
  MPF = MPF + i*xf[frequency_pointer]
  frequency_pointer += 1

whole_MPF = MPF / fullSum

print('The mean power frequency of the entire dataset is', whole_MPF,'Hz',)


# 5. After understanding how to use FFT, MF and MPF, try to process the EMG signal bit by bit!

> Taking a 2 second segment of the EMG signal and calculate the MF and MPF of this segment!

> After that shift back 0.5 seconds and reexecute the code. Observe changes of MF and MPF after recalculation!

In [ ]:
loop_time = 500
loop_number = int((data3.size-N)/loop_time) #Calculate how many times do we loop through the code if we execute it every 0.5 seconds
loop_counter = 0 #Every 0.5 second execution +1

N = 2000 #Grab 2000 data points to do FFT (for data of 1000Hz, 1 second of data gives 1000 data points)
T = 0.001 #The interval between each data point is 0.001 second (for data of 1000Hz, data point interval is 0.001 second)
x = np.linspace(0.0, N*T, N, endpoint=False)
xf = fftfreq(N, T)[1:N//2]

MPF_set = np.zeros(loop_number) #Saves the MPF data for every 0.5 second interval
MF_set = np.zeros(loop_number)  #Saves the MF data for every 0.5 second interval


while loop_counter < loop_number:

  #-------------FFT---------------#
  yf = fft(data3[1+(loop_time)*(loop_counter):N+(loop_time)*(loop_counter)])
  yf_abs = np.abs(yf[1:N//2])

  #-------------MF---------------#
  halfSum = 0 #Prepare to sum up to half of the total energy
  MF = 0 #Calculate the position of MF in the frequency domain
  fullSum = np.sum(yf_abs) #Sum of all the energy

  for i in yf_abs:
    if(halfSum < fullSum/2): #When we add up to halfSum we arrive at MF's position (half of total energy)
      halfSum = halfSum + i
      MF += 1
    else:
      break
  MF_set[loop_counter] = MF


  #-------------MPF---------------#
  frequency_pointer = 0 #Pointer to get the frequency domain data
  MPF = 0 #Prepare to sum up to total energy

  for i in yf_abs:
    MPF = MPF + i*xf[frequency_pointer]
    frequency_pointer += 1

  MPF = MPF / fullSum

  MPF_set[loop_counter] = MPF

  #-------------Finish running 1 set of 0.5 second---------------#
  #---------------Start the next set!---------------#
  loop_counter+=1
  #....
  #............
  #....................
  #........................
  #------Run until loop_counter equals loop_number-------#

print("Successfully analysed the MF and MPF of the entire EMG signal!")

# 6. Let's see the changes to MF and MPF after our analysis and use it to estimate the muscle fatigue!

In [ ]:
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=MF_set, #Plot out changes to MF
    mode='lines',
    name='MF Plot',
)
)
fig.add_trace(go.Scatter(
    y=MPF_set, #Plot out changes to MPF
    mode='lines',
    name='MPF Plot',
)
)
fig.update_layout(
    title="Entire EMG signal of when muscle is tensioned's MF and MPF changes",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)

part_MF_max = 0
part_MF_min = 1000
part_MPF_max = 0
part_MPF_min = 1000

for i in MF_set:
  if i>part_MF_max:
    part_MF_max = i
  if i<part_MF_min:
    part_MF_min = i
for i in MPF_set:
  if i>part_MPF_max:
    part_MPF_max = i
  if i<part_MPF_min:
    part_MPF_min = i

print("Unsegmented EMG's MF:",int(whole_MF))
print("Unsegmented EMG's MPF:",int(whole_MPF))
print("Segmented EMG's largest MF:",int(part_MF_max))
print("Segmented EMG's smallest MF:",int(part_MF_min))
print("Segmented EMG's largest MPF:",int(part_MPF_max))
print("Segmented EMG's smallest MPF:",int(part_MPF_min))
fig.show()

# Chapter Summary
---

1.   Even though EMG signal's frequency domain analysis have a clear guideline academically. To actually cause muscle fatigue that are reflected on the EMG signal clearly, you can't just tension your muscle once.
2.   Even though MF and MPF have a completely different formula, but the physical significance of both is to reflect the energy change for medium to low frequency. Hence both parameters's value follow a similar trend.
3.   Is it possible that after tensioning your muscle multiple times, there is a clearer evidence of muscle fatigue by comparing the first few measurement with last few (such as lower average MF or average MPF)?

# Congratulation on completing this chapter's classes



> For student that needs the class certificate you can use the password on the bookstore website to obtain it

> Certificate password: [yutechEMG2]!